# Web Crawler Demo: linktrace with Callbacks (Memory-Efficient)

This notebook demonstrates the linktrace Spider using callbacks with `accumulate_results=False` to process results as they're crawled without accumulating them in memory.

**Comparison:** Compare memory usage with `crawl_cnn.ipynb` which accumulates all results in memory.

In [ ]:
import json
import logging
from collections import Counter
from datetime import datetime

import psutil

from linktrace import Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

# Get initial memory usage
process = psutil.Process()
initial_memory = process.memory_info().rss / 1024 / 1024  # MB
print(f"Initial memory: {initial_memory:.2f} MB")

## Stream URLs with Callbacks (Memory-Efficient)

Define callbacks to process and display results as they're crawled, without keeping them in memory.

In [ ]:
# Track crawl statistics
crawl_stats = {
    "total_pages": 0,
    "total_internal_links": 0,
    "total_external_links": 0,
    "failed_urls": [],
}

# Store first 10 URLs for display
urls_crawled = []


def on_page_crawled(doc):
    """Called after each page is crawled - process and discard."""
    crawl_stats["total_pages"] += 1
    crawl_stats["total_internal_links"] += len(doc.internal_links)
    crawl_stats["total_external_links"] += len(doc.external_links)

    # Store first 10 URLs for display
    if len(urls_crawled) < 10:
        urls_crawled.append(
            {
                "url": doc.url,
                "title": doc.title[:60],
                "status": doc.status_code,
                "internal_links": len(doc.internal_links),
                "external_links": len(doc.external_links),
            }
        )

    # Print progress every 10 pages
    if crawl_stats["total_pages"] % 10 == 0:
        current_memory = process.memory_info().rss / 1024 / 1024
        print(
            f"  Crawled {crawl_stats['total_pages']} pages | "
            f"Memory: {current_memory:.2f} MB | "
            f"Last: {doc.url}"
        )

    # Don't return anything - results are discarded
    return None


def on_error(url, exception):
    """Called when a page fails to crawl."""
    crawl_stats["failed_urls"].append({"url": url, "error": str(exception)})
    print(f"  ❌ Failed: {url}")


def on_crawl_complete():
    """Called when crawl finishes."""
    print(f"\n✓ Crawl complete!")


# Run spider with callbacks - accumulate_results=False means no memory buildup
spider = Spider(
    start_url="https://www.cnn.com",
    max_depth=1,
    on_page_crawled=on_page_crawled,
    on_error=on_error,
    on_crawl_complete=on_crawl_complete,
    accumulate_results=False,  # KEY: Don't accumulate in memory
    show_progress=True,
)

print(f"Starting streaming crawl with max_depth=3...\n")
result = await spider.run_async()

print(f"\nResult list: {len(result)} (empty because accumulate_results=False)")

## Crawl Statistics

In [ ]:
# Calculate final memory usage
final_memory = process.memory_info().rss / 1024 / 1024
memory_change = final_memory - initial_memory

print("=" * 70)
print("CRAWL STATISTICS (Streaming with Callbacks)")
print("=" * 70)
print(f"Total pages crawled: {crawl_stats['total_pages']}")
print(f"Total internal links found: {crawl_stats['total_internal_links']}")
print(f"Total external links found: {crawl_stats['total_external_links']}")
print(f"Failed URLs: {len(crawl_stats['failed_urls'])}")
print()
print(f"Initial memory: {initial_memory:.2f} MB")
print(f"Final memory: {final_memory:.2f} MB")
print(f"Memory change: {memory_change:+.2f} MB")
print(f"Avg memory per page: {memory_change / crawl_stats['total_pages']:.4f} MB")
print()
print(
    f"Avg links per page: "
    f"{(crawl_stats['total_internal_links'] + crawl_stats['total_external_links']) / crawl_stats['total_pages']:.1f}"
)

## First 10 URLs Crawled

In [ ]:
print(f"{'URL':<50} {'Title':<20} {'Status':<8} {'Links':<8}")
print("-" * 90)

for item in urls_crawled:
    url_short = item["url"][:49]
    title_short = item["title"][:19]
    total_links = item["internal_links"] + item["external_links"]
    print(f"{url_short:<50} {title_short:<20} {item['status']:<8} {total_links:<8}")

## Failed URLs (if any)

In [ ]:
if crawl_stats["failed_urls"]:
    print(f"Failed URLs ({len(crawl_stats['failed_urls'])}):")
    for failed in crawl_stats["failed_urls"]:
        print(f"  • {failed['url']}")
        print(f"    Error: {failed['error'][:60]}...")
else:
    print("No failed URLs!")

## Comparison: Streaming vs Accumulating

**This notebook (Streaming):**
- ✅ Results processed as pages are crawled
- ✅ No results kept in memory
- ✅ Predictable, minimal memory growth
- ✅ Perfect for large crawls

**crawl_cnn.ipynb (Accumulating):**
- ❌ All pages accumulated in `documents` list
- ❌ Memory grows linearly with page count
- ❌ Can cause OOM on very large crawls
- ✅ Convenient for analysis at end

**For large crawls (1000+ pages):** Use streaming callbacks (`accumulate_results=False`)